<a href="https://colab.research.google.com/github/soham-never-codes/Festiva-AI-Event-Planner/blob/main/notebooks/Festiva_Week5_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  Mount + install + setup ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT_DIR = '/content/drive/MyDrive/festiva'

# The essential safety net
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/rag', exist_ok=True)

print("⬇️ Installing necessary libraries...")
!pip install -q openai langchain langchain-community sentence-transformers faiss-cpu joblib
print("✅ Environment ready!")

In [ ]:
# ── CELL 2: Set your API key (securely) ──────────────────────────────────
import os
from google.colab import userdata

# [UPDATED] Pulling the Gemini key
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print("✅ Gemini API key loaded securely from Colab Secrets!")

In [ ]:
#  Load all saved components ────────────────────────────────────
import joblib
import numpy as np
import torch
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
import re

# Safely check for GPU
current_device = "cuda" if torch.cuda.is_available() else "cpu"
current_device_id = 0 if torch.cuda.is_available() else -1
print(f"⚙️ Hardware detected: {current_device.upper()}")

# 1. Load ML budget model
model    = joblib.load(f'{PROJECT_DIR}/models/budget_model.pkl')
scaler   = joblib.load(f'{PROJECT_DIR}/models/scaler.pkl')
encoders = joblib.load(f'{PROJECT_DIR}/models/encoders.pkl')
meta     = joblib.load(f'{PROJECT_DIR}/models/metadata.pkl')

FEATURE_COLS = meta['feature_cols']
SPEND_COLS   = meta['spend_cols']

# 2. Load RAG Vector Database
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": current_device}
)
vectorstore = FAISS.load_local(
    f'{PROJECT_DIR}/rag/faiss_index', embedder,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Load NLP classifier
nlp_clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=current_device_id
)

print("✅ All components successfully loaded into memory!")

In [ ]:
#  Define agent functions ───────────────────────────────────────
import re
import numpy as np
import json
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Force Gemini to return perfectly structured JSON
llm = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config={"response_mime_type": "application/json"}
)

# Agent 1: NLP — parse the query
CITIES = ["bangalore","bengaluru","mumbai","delhi","chennai","hyderabad","pune"]
CITY_NORM = {"bengaluru":"Bangalore"}

def nlp_agent(text):
    entities = {}
    patterns = [(r'₹\s*(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)', 1e5),
                (r'(\d+(?:\.\d+)?)\s*(?:L\b|lakh|lakhs)', 1e5),
                (r'₹\s*([\d,]+)', 1)]
    for pat, mult in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            entities['budget'] = int(float(m.group(1).replace(',','')) * mult)
            break
    for city in CITIES:
        if city in text.lower():
            entities['city'] = CITY_NORM.get(city, city.capitalize())
            break
    m = re.search(r'(\d+)\s+(?:guests?|people|attendees?)', text, re.IGNORECASE)
    if m:
        entities['guest_count'] = int(m.group(1))

    labels = ["wedding","corporate event","birthday party","engagement","conference"]
    res = nlp_clf(text, labels)
    raw = res['labels'][0]
    etype = raw.replace(' party','').replace(' event','').replace(' ','_')

    return {
        'event_type': entities.get('event_type', etype),
        'city':       entities.get('city', 'Bangalore'),
        'budget':     entities.get('budget', 500_000),
        'guests':     entities.get('guest_count', 100)
    }

# Agent 2: Budget optimizer
def budget_agent(event_type, city, budget, guests, duration=1, season='winter'):
    try:
        row = np.array([[
            encoders['event_type'].transform([event_type])[0],
            encoders['city'].transform([city])[0],
            guests, budget, duration,
            encoders['season'].transform([season])[0]
        ]])
    except Exception:
        row = np.array([[0, 0, guests, budget, duration, 0]])

    pcts = model.predict(scaler.transform(row))[0]
    pcts = np.clip(pcts, 0, None)
    pcts /= pcts.sum()

    return {c: {'pct': round(p*100,1), 'amount': int(p*budget)}
            for c, p in sorted(zip(SPEND_COLS, pcts), key=lambda x: -x[1])}

# Agent 3: Knowledge retriever
def knowledge_agent(event_type, city, budget):
    query = f"{event_type} planning guide {city}"
    docs = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)

# Agent 4: Planner LLM (Now powered by Gemini!)
def planner_agent(query, parsed, budget_breakdown, context):
    budget_str = "\n".join(
        f"  {k}: ₹{v['amount']:,} ({v['pct']}%)"
        for k, v in budget_breakdown.items()
    )

    prompt = f"""
    You are Festiva, an expert Indian event planner.
    Respond ONLY with a valid JSON object using EXACTLY these keys:
    "summary" (a string summarizing the plan),
    "timeline" (a list of objects, each with a 'week' string and a 'tasks' list of strings),
    "checklist" (a list of strings),
    "vendor_categories" (a list of objects, each with 'category', 'priority', and 'tip' strings).

    User Request: {query}
    Detected Parameters: {parsed}

    Optimal Budget Breakdown (from ML model):
    {budget_str}

    Knowledge Base Guidelines:
    {context[:1500]}
    """

    response = llm.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
# Orchestrator — ties everything together ───────────────────────
def festiva_plan(user_query):
    print(f"\n🚀 Processing Request: '{user_query}'\n")
    print("-" * 50)

    # Step 1: NLP
    parsed = nlp_agent(user_query)
    print(f"🧠 [Agent 1] NLP detected: {parsed}")

    # Step 2: Budget ML
    breakdown = budget_agent(
        parsed['event_type'], parsed['city'],
        parsed['budget'], parsed['guests']
    )
    print(f"📊 [Agent 2] ML Model: Generated optimal budget for {len(breakdown)} categories")

    # Step 3: RAG
    context = knowledge_agent(parsed['event_type'], parsed['city'], parsed['budget'])
    print(f"📚 [Agent 3] RAG DB: Retrieved {len(context)} characters of planning context")

    # Step 4: LLM planner
    plan = planner_agent(user_query, parsed, breakdown, context)
    print("🤖 [Agent 4] LLM: Master plan generated successfully!")
    print("-" * 50)

    return {
        "input": user_query,
        "detected": parsed,
        "budget_breakdown": breakdown,
        "plan": plan
    }

In [ ]:
#  Run end-to-end test ───────────────────────────────────────────
result = festiva_plan("Wedding in Bangalore, budget ₹10L, 200 guests")

print("\n" + "=" * 60)
print(" 🎊 FESTIVA EVENT PLAN")
print("=" * 60)
print(f"\n📝 Summary: {result['plan'].get('summary','')}")

print("\n💰 Budget Breakdown:")
for cat, v in result['budget_breakdown'].items():
    if v['pct'] > 0:  # Only show relevant categories
        bar = '█' * int(v['pct'] / 2)
        print(f"  {cat.capitalize():<22} ₹{v['amount']:>8,}  {v['pct']:>5.1f}%  {bar}")

print("\n📅 Timeline:")
for item in result['plan'].get('timeline', []):
    print(f"  📌 {item.get('week','')}: {', '.join(item.get('tasks',[]))}")

print("\n✅ Top 5 Checklist Items:")
for i, item in enumerate(result['plan'].get('checklist', [])[:5], 1):
    print(f"  {i}. {item}")

print("\n🤝 Vendor Strategies:")
for v in result['plan'].get('vendor_categories', []):
    print(f"  [{v.get('priority','')}] {v.get('category','')} — {v.get('tip','')}")